# Метод максимального правдоподобия
### Как ML «угадывает» параметры модели

---

Представьте: вы видите следы на снегу и пытаетесь понять, чьи они.
Вы не можете проверить всех зверей — но можете спросить:

> **«При каком звере эти следы были бы наиболее вероятны?»**

Именно так работает метод максимального правдоподобия.

$$ L(\theta) = \prod_{i=1}^{n} f(x_i \mid \theta) \to \max_{\theta} $$

**(MLE)** — это механизм, который стоит внутри почти всех современных моделей искусственного интеллекта.



Метод максимального правдоподобия — способ, когда мы смотрим на то, что **УЖЕ** произошло (на данные), и выбираем самое логичное объяснение из всех возможных.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy import stats

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
GREEN  = '#55A868'
RED    = '#C44E52'
PURPLE = '#9467bd'
GRAY   = '#888888'


---
## Что такое ММП (MLE)&

Допустим, у нас есть монетка. Мы не знаем, честная ли она. То есть вероятности мы не знаем
Бросаем **10 раз** — выпадает **7 орлов**.

**Вопрос:** какова вероятность орла у этой монеты?

Интуитивный ответ — **7/10 = 0.7**. Но почему именно 0.7, а не 0.5 или 0.9?

Давайте посмотрим на это как на график.

In [ ]:
p_vals = np.linspace(0.01, 0.99, 400)
n, k = 10, 7

likelihood = p_vals**k * (1 - p_vals)**(n - k)
likelihood /= likelihood.max()

p_mle = k / n

fig, ax = plt.subplots(figsize=(10, 5))

ax.fill_between(p_vals, likelihood, alpha=0.15, color=BLUE)
ax.plot(p_vals, likelihood, color=BLUE, lw=3)

for p_check, label_offset in [(0.3, 0.07), (0.5, 0.07), (0.7, -0.12), (0.9, 0.07)]:
    lval = p_check**k * (1 - p_check)**(n - k)
    lval /= (p_mle**k * (1 - p_mle)**(n - k))
    color = RED if p_check == p_mle else GRAY

ax.axvline(p_mle, color=RED, lw=2, ls='--')
ax.set_xlabel('Предполагаемая вероятность орла (p)', fontsize=13)
ax.set_ylabel('Насколько это «правдоподобно»', fontsize=13)
ax.set_title('При каком p наши данные (7 орлов из 10) наиболее вероятны?',
             fontsize=13, fontweight='bold')

ax.text(0.71, 0.5, '← максимум\n   здесь!\n   p = 0.7', fontsize=11,
        color=RED, fontweight='bold')

ax.set_yticks([])
plt.tight_layout()
plt.show()

### Что мы только что сделали?

Мы перебрали все возможные значения параметра **p** от 0 до 1
и для каждого спросили: *«Насколько вероятно, что при таком p мы бы увидели наши данные?»*

Точка, где этот вопрос имеет **максимальный ответ** — это и есть **оценка методом максимального правдоподобия (MLE)**.

---

Для монетки ответ очевиден: `p = 7/10 = 0.7`.
Но в ML параметров могут быть **миллионы**. Там нужен более умный способ найти максимум.

---
## MLE в линейной регрессии

Задача: предсказать цену квартиры по площади.

Модель рисует прямую. Но как выбрать **лучшую прямую** из бесконечного числа вариантов?

MLE отвечает: ту, при которой наши данные **наиболее правдоподобны**.

In [ ]:
np.random.seed(42)
n_pts = 25
x = np.linspace(20, 80, n_pts)
y = 1.8 * x + 30 + np.random.normal(0, 12, n_pts)

w = np.polyfit(x, y, 1)
y_pred = np.polyval(w, x)

lines = [
    (0.5,  60,  ORANGE, 'Плохая прямая'),
    (w[0], w[1], RED,   f'MLE: лучшая прямая'),
    (3.5,  -10, PURPLE, 'Плохая прямая'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, (slope, intercept, color, title) in zip(axes, lines):
    y_line = slope * x + intercept
    residuals = y - y_line
    mse = np.mean(residuals**2)

    for xi, yi, ypi in zip(x, y, y_line):
        ax.plot([xi, xi], [yi, ypi],
                color='tomato' if color != RED else GREEN,
                lw=1.5, alpha=0.7)

    ax.scatter(x, y, color=BLUE, s=50, zorder=5, label='Данные')
    ax.plot(x, y_line, color=color, lw=2.5, label=title)

    err_color = GREEN if color == RED else 'tomato'
    ax.set_title(title, fontsize=11, fontweight='bold', color=color)
    ax.set_xlabel('Площадь, м²', fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel('Цена, тыс. руб.', fontsize=11)


plt.suptitle('MLE выбирает прямую, при которой ошибки наименьшие',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Почему логарифм?

Чтобы найти максимум, нам нужно перемножить правдоподобие по всем точкам датасета.

Представьте: у вас **1000 точек**, и каждая даёт число вроде `0.03`.

Произведение: $0.03 × 0.03 × \ldots$ (1000 раз) → число настолько маленькое, что компьютер пишет `0.0`.

$$ \ln L = \sum_{i=1}^{n} \ln f(x_i \mid \theta) \to \max_{\theta} $$

In [ ]:
p_small = 0.03
result_100  = p_small ** 100
result_1000 = p_small ** 1000

print(f'0.03 в степени 100  = {result_100:.2e}')
print(f'0.03 в степени 1000 = {result_1000}')
print()
print('Но логарифм превращает произведение в сумму')
print(f'log(0.03) × 100 = {np.log(p_small) * 100:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))


p_range = np.linspace(0.01, 0.99, 300)
L   =  p_range**7 * (1 - p_range)**3
logL = 7*np.log(p_range) + 3*np.log(1 - p_range)

L_norm    = (L - L.min())    / (L.max()    - L.min())
logL_norm = (logL - logL.min()) / (logL.max() - logL.min())

ax.plot(p_range, L_norm,    color=BLUE,  lw=2.5)
ax.plot(p_range, logL_norm, color=GREEN, lw=2.5, ls='--')
ax.axvline(0.7, color=RED, lw=2, ls=':', label='Максимум: p = 0.7')
ax.scatter([0.7], [1.0], color=RED, s=100, zorder=6)
ax.set_xlabel('p', fontsize=12)
ax.set_ylabel('Значение (нормализованное)', fontsize=11)
ax.set_title('Логарифм монотонен, т.е. не сдвигает максимум!\nОтвет остаётся тем же', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## Тёмная сторона MLE — переобучение

MLE — честный метод. Он делает ровно то, что ему говорят:
> **«Сделай так, чтобы эти данные были как можно более вероятны»**

На маленьких данных это опасно — модель «заучивает» конкретные точки,
а не находит реальную зависимость.

Иными словами, мы натыкаемся на феномен **переобучения**

In [ ]:
np.random.seed(0)
n_small = 12
x_tr = np.sort(np.random.uniform(0, 1, n_small))
y_tr = np.sin(2 * np.pi * x_tr) + np.random.normal(0, 0.25, n_small)

x_vis = np.linspace(0, 1, 300)
y_true = np.sin(2 * np.pi * x_vis)

configs = [
    (2,  BLUE,   'Простая модель\n(недообучение)'),
    (5,  GREEN,  'Хорошая модель\n'),
    (11, RED,    'Сложная модель\n(переобучение — MLE перестарался)')
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (deg, color, title) in zip(axes, configs):
    coeffs = np.polyfit(x_tr, y_tr, deg)
    y_fit  = np.clip(np.polyval(coeffs, x_vis), -3, 3)
    train_err = np.mean((np.polyval(coeffs, x_tr) - y_tr)**2)

    ax.plot(x_vis, y_true, color=GRAY, lw=2, ls='--', alpha=0.7)
    ax.plot(x_vis, y_fit,  color=color, lw=2.5)
    ax.scatter(x_tr, y_tr, color='black', s=60, zorder=5)

    ax.set_ylim(-2.2, 2.2)
    ax.set_title(title, fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('x')
    if ax == axes[0]:
        ax.set_ylabel('y')
    ax.text(0.03, 1.6, f'Ошибка на обучении: {train_err:.3f}',
            fontsize=8.5, color='#555')


plt.suptitle('MLE переобучается: он минимизирует ошибку на тренировочных данных,\nне думая о том, что будет на новых',
             fontsize=12, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

---
## Решение — регуляризация

Идея простая: добавим **штраф за сложность**.

Модель теперь хочет две вещи одновременно:
1. Хорошо объяснить данные
2. Оставаться **простой**

Параметр **λ (лямбда)** управляет балансом между этими двумя целями.

In [ ]:
from sklearn.linear_model import Ridge

deg = 11
X_tr_poly  = np.vander(x_tr,  deg+1)
X_vis_poly = np.vander(x_vis, deg+1)

alphas = [
    (0,     RED,    'MLE без ограничений\n(λ = 0)'),
    (1e-4,  ORANGE, 'Лёгкий штраф\n(λ = 0.0001)'),
    (0.1,   GREEN,  'Хороший штраф\n(λ = 0.1)'),
    (10,    BLUE,   'Слишком большой штраф\n(λ = 10)'),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, (alpha, color, title) in zip(axes, alphas):
    if alpha == 0:
        coeffs = np.linalg.lstsq(X_tr_poly, y_tr, rcond=None)[0]
        y_fit = X_vis_poly @ coeffs
    else:
        ridge = Ridge(alpha=alpha, fit_intercept=False)
        ridge.fit(X_tr_poly, y_tr)
        y_fit = ridge.predict(X_vis_poly)

    y_fit = np.clip(y_fit, -2.2, 2.2)

    ax.plot(x_vis, y_fit,  color=color, lw=2.5)
    ax.scatter(x_tr, y_tr, color='black', s=55, zorder=5)

    ax.set_ylim(-2.2, 2.2)
    ax.set_title(title, fontsize=9.5, fontweight='bold', color=color)
    ax.set_xlabel('x')

plt.tight_layout()
plt.show()

## Контрольные вопросы

1. Если в нашем примере заменить 10 бросков и 7 орлов на 10000 бросков и 7000 орлов, точка максимума поменяется?
А значение функции?
2. Дайте математическое определенеи ММП
3. Почему в примере с монеткой мы пишем именно p**7 * (1-p)**3, а не что-то другое?
4. Почему обычное правдоподобие превращается почти в ноль при большом количестве данных, а логарифмическое остаётся нормальным числом?
5. Почему мы логарифмируем нашу функцию? Причем здесь производная?
6. Зачем нужен ММП (MLE) в ML? Какую функцию он выполняет?
7. Какие есть риски при использовании MLE?

## Ответы на контрольные вопросы

1. Точка максимума не поменяется: и для 7/10, и для 7000/10000 MLE даёт `p = k / n = 0.7`. Значение функции правдоподобия поменяется: при 10000 бросках оно станет гораздо меньше по абсолютной величине и кривая будет намного острее около максимума.

2. ММП — это выбор такого параметра `theta`, который максимизирует правдоподобие наблюдаемых данных:
   `theta_hat = argmax_theta L(theta) = argmax_theta prod_i f(x_i | theta)`.
   Эквивалентно часто максимизируют лог-правдоподобие:
   `theta_hat = argmax_theta sum_i log f(x_i | theta)`.

3. В примере с монеткой используется модель Бернулли: вероятность орла равна `p`, вероятность решки равна `1 - p`. Если в 10 независимых бросках выпало 7 орлов и 3 решки, то вероятность такой последовательности пропорциональна `p**7 * (1-p)**3`.

4. Обычное правдоподобие является произведением большого числа вероятностей, каждая из которых меньше или равна 1. При большом числе наблюдений произведение быстро становится чрезвычайно маленьким и может численно округлиться до нуля. Логарифм заменяет произведение суммой логарифмов, поэтому значения остаются вычислительно устойчивыми.

5. Мы логарифмируем функцию, потому что логарифм монотонно возрастает и не меняет точку максимума, но сильно упрощает вычисления. Произведения превращаются в суммы, степени — в множители. Производная нужна, чтобы найти максимум: приравнять производную лог-правдоподобия к нулю и получить оптимальный параметр.

6. MLE нужен в ML, чтобы подбирать параметры модели так, чтобы наблюдаемые данные были максимально правдоподобны при выбранной вероятностной модели. Он фактически задаёт функцию обучения: максимизировать likelihood или, что то же самое, минимизировать отрицательное лог-правдоподобие.

7. Риски MLE: переобучение на обучающих данных, особенно у слишком сложных моделей; чувствительность к выбросам и неверным предположениям о распределении шума; численная нестабильность без логарифмирования; плохая обобщающая способность без регуляризации, валидации и достаточного объёма данных.
